# Batch FASTA, MSA, and Group Enrichment Workflow

This notebook keeps the heavier batch steps in one place. Metadata is optional.

In [ ]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from Bio import SeqIO
from aaseq.alignment import conservation_dataframe, consensus_sequence, reference_msa
from aaseq.enrichment import group_feature_enrichment
from aaseq.processing import process_fasta_to_matrix
from aaseq.qc import analyze_fasta_qc
from aaseq.visualization import plot_conservation, visualize_matrix

fasta_path = "examples/ncbi/example_proteins.fasta"
metadata_path = None  # e.g. "metadata.csv" with columns id,group
min_len, max_len, min_reps = 3, 10, 2

In [ ]:
qc_summary, qc_issues = analyze_fasta_qc(fasta_path)
qc_summary, qc_issues.head()

In [ ]:
matrix = process_fasta_to_matrix(
    fasta_path,
    min_len=min_len,
    max_len=max_len,
    min_reps=min_reps,
    workers=2,
    include_fuzzy=True,
    fuzzy_mismatches=1,
    include_idr=True,
)
matrix.head()

In [ ]:
motif_cols = [c for c in matrix.columns if c.startswith("Fuzzy_") or set(c).issubset(set("ACDEFGHIKLMNPQRSTVWY"))]
if motif_cols:
    visualize_matrix(matrix[motif_cols].iloc[:, :50], title="Repeat and Fuzzy Motif Features")

In [ ]:
records = list(SeqIO.parse(fasta_path, "fasta"))
alignment = reference_msa([r.id for r in records], [str(r.seq) for r in records])
conservation = conservation_dataframe(alignment)
consensus = consensus_sequence(alignment)
consensus[:120], conservation.head()

In [ ]:
plot_conservation(conservation)

In [ ]:
if metadata_path:
    enrichment = group_feature_enrichment(matrix, metadata_path, id_col="id", group_col="group")
    display(enrichment.head(30))